In [ ]:
"""
Episode start/end intention labeling using a weighted keyword dictionary.

Reads:
  - Episodes CSV (All_episodes_with_messages.csv)
  - Dictionary file (dictionary.xlsx OR dictionary.csv) in wide format:
      one column per label, one keyword per cell
    Optional weight columns per label:
      <LabelName>__weight  (numeric)

Writes (to out_dir):
  - episodes_with_intentions.csv
  - start_intent_candidate_keywords.csv
  - end_intent_candidate_keywords.csv
"""

from __future__ import annotations

import argparse
import re
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer("english")

# ---- Columns expected in All_episodes_with_messages.csv ----
START_FIELDS = ["start_commit_subject", "start_commit_body", "start_pr_title", "start_pr_body", "start_issue_summary"]
END_FIELDS   = ["end_commit_subject", "end_commit_body", "end_pr_title", "end_pr_body", "end_issue_summary"]

# Minimal stop-list for candidate expansion
STOP = set("""
a an the and or but if then else when while of to for in on at by from as is are was were be been being with without
this that these those it its i you we they he she them his her our your their into over under up down out off via vs
add adds added adding remove removed removing fix fixes fixed fixing update updates updated updating change changes changed changing
refactor refactors refactored refactoring bump bumps bumped merge merges merged revert reverts reverted
""".split())

# --------- DEFAULT WINDOWS PATHS (YOUR LOCATIONS) ----------
DEFAULT_BASE_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3_2")
DEFAULT_OUT_DIR  = DEFAULT_BASE_DIR / "Intention_Detection"
DEFAULT_EPISODES_PATH = DEFAULT_BASE_DIR / "All_episodes_with_messages.csv"

# Prefer xlsx, else csv (auto-detected in loader)
DEFAULT_DICT_CSV  = DEFAULT_OUT_DIR / "dictionary.csv"
# -----------------------------------------------------------


def normalize_text_for_tokens(text: str) -> str:
    # de-camelcase + underscores/hyphens -> space
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    text = text.replace("_", " ").replace("-", " ")
    return text.lower()


def tokenize_and_stem(text: str) -> List[str]:
    if not text:
        return []
    text = normalize_text_for_tokens(text)
    tokens = re.findall(r"[a-z]+", text)
    return [stemmer.stem(t) for t in tokens]


def combine_fields(row: pd.Series, fields: List[str]) -> str:
    parts: List[str] = []
    for f in fields:
        if f not in row:
            continue
        v = row.get(f, "")
        if pd.isna(v) or v is None:
            continue
        s = str(v).strip()
        if s:
            parts.append(s)
    return "\n".join(parts)


def count_phrase_occurrences(tokens: List[str], phrase_tokens: List[str]) -> int:
    if not phrase_tokens or not tokens:
        return 0
    if len(phrase_tokens) == 1:
        return sum(1 for t in tokens if t == phrase_tokens[0])
    n = len(phrase_tokens)
    cnt = 0
    for i in range(len(tokens) - n + 1):
        if tokens[i : i + n] == phrase_tokens:
            cnt += 1
    return cnt


def _read_dictionary_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Dictionary file not found: {path}")

    suf = path.suffix.lower()
    if suf in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if suf == ".csv":
        return pd.read_csv(path)

    raise ValueError(f"Unsupported dictionary format: {path} (use .xlsx/.xls or .csv)")


def load_dictionary_wide(path: Path) -> Dict[str, List[Dict]]:
    df = _read_dictionary_table(path)

    # Identify label columns (exclude 'no' and weight columns)
    if "no" in df.columns:
        labels = [c for c in df.columns if c != "no" and not str(c).endswith("__weight")]
    else:
        labels = [c for c in df.columns if not str(c).endswith("__weight")]

    if not labels:
        raise ValueError(f"No label columns found in dictionary: {path}")

    dict_terms: Dict[str, List[Dict]] = {label: [] for label in labels}

    for label in labels:
        wcol = f"{label}__weight"
        weights = df[wcol] if wcol in df.columns else None

        # Some dictionaries may have missing label column entirely in CSV
        if label not in df.columns:
            continue

        for idx, cell in df[label].items():
            if pd.isna(cell):
                continue
            cell_s = str(cell).strip()
            if not cell_s:
                continue

            # Allow multiple keywords in one cell: "foo; bar, baz"
            for kw in re.split(r"[;,/|]", cell_s):
                kw = kw.strip()
                if not kw:
                    continue

                wt = 1.0
                if weights is not None:
                    wv = weights.iloc[idx]
                    if not pd.isna(wv):
                        try:
                            wt = float(wv)
                        except Exception:
                            wt = 1.0

                stem_tokens = tokenize_and_stem(kw)
                if not stem_tokens:
                    continue

                dict_terms[label].append({"keyword": kw, "weight": wt, "stem_tokens": stem_tokens})

        # de-dup within label by stem sequence
        seen = set()
        uniq = []
        for item in dict_terms[label]:
            key = " ".join(item["stem_tokens"])
            if key in seen:
                continue
            seen.add(key)
            uniq.append(item)
        dict_terms[label] = uniq

    return dict_terms


def classify_text(
    text: str,
    dict_terms: Dict[str, List[Dict]],
    cap_per_keyword: bool = True
) -> Tuple[Dict[str, float], Dict[str, List[str]]]:
    tokens = tokenize_and_stem(text or "")
    scores = {label: 0.0 for label in dict_terms}
    matches = {label: [] for label in dict_terms}

    for label, items in dict_terms.items():
        for it in items:
            occ = count_phrase_occurrences(tokens, it["stem_tokens"])
            if occ > 0:
                if cap_per_keyword:
                    occ = 1
                scores[label] += it["weight"] * occ
                matches[label].append(it["keyword"])

    return scores, matches


def assign_labels_multi(
    scores: Dict[str, float],
    matches: Dict[str, List[str]],
    min_score: float = 1.0,
    multi_ratio: float = 0.8,
    max_labels: int = 3,
) -> Tuple[str, float, str]:
    """
    Returns:
      labels_str: "A" or "A || B" (multi-label)
      confidence: top_score / sum_scores
      matched_keywords: keywords matched in chosen labels
    """
    items = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    top_label, top_score = items[0]
    if top_score < min_score:
        return "UNCLASSIFIED", 0.0, ""

    chosen = []
    for lab, sc in items:
        if sc < min_score:
            break
        if sc >= top_score * multi_ratio:
            chosen.append(lab)
        if len(chosen) >= max_labels:
            break

    total = sum(scores.values())
    conf = (top_score / total) if total > 0 else 1.0
    matched = sorted({kw for lab in chosen for kw in matches.get(lab, [])})
    return " || ".join(chosen), conf, "; ".join(matched)


def get_primary(label_str: str) -> str:
    return (label_str or "").split("||")[0].strip()


def build_candidate_suggestions(
    df: pd.DataFrame,
    text_fields: List[str],
    label_col: str,
    dict_terms: Dict[str, List[Dict]],
    top_n: int = 200,
    min_freq: int = 5,
    ratio: float = 1.5,
    min_assoc: int = 3,
) -> pd.DataFrame:
    # Dictionary stems to exclude (single token only)
    dict_stems = set()
    for _, items in dict_terms.items():
        for it in items:
            if len(it["stem_tokens"]) == 1:
                dict_stems.add(it["stem_tokens"][0])

    # 1) candidate tokens from UNCLASSIFIED
    cand_counts = defaultdict(int)
    uncls = df[df[label_col] == "UNCLASSIFIED"]
    for _, row in uncls.iterrows():
        text = combine_fields(row, text_fields)
        tokens = tokenize_and_stem(text)
        for t in tokens:
            if t in STOP or len(t) < 3:
                continue
            if t in dict_stems:
                continue
            cand_counts[t] += 1

    cands = [(t, c) for t, c in cand_counts.items() if c >= min_freq]
    cands.sort(key=lambda x: x[1], reverse=True)
    cands = cands[:top_n]

    # 2) association of candidates with already-classified (primary label)
    labeled = df[df[label_col] != "UNCLASSIFIED"].copy()
    labeled["primary"] = labeled[label_col].astype(str).map(get_primary)

    labeled_tokens = []
    for _, row in labeled.iterrows():
        labeled_tokens.append(set(tokenize_and_stem(combine_fields(row, text_fields))))

    labels = list(dict_terms.keys())
    suggestions = []
    for tok, freq in cands:
        counts = {lab: 0 for lab in labels}
        for prim, toks in zip(labeled["primary"].tolist(), labeled_tokens):
            if tok in toks and prim in counts:
                counts[prim] += 1

        items = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)
        top_lab, top_c = items[0]
        top2_lab, top2_c = items[1] if len(items) > 1 else ("", 0)
        max_other = max([c for lab, c in counts.items() if lab != top_lab], default=0)
        max_rest = max([c for lab, c in counts.items() if lab not in {top_lab, top2_lab}], default=0)

        # Strong single-label signal -> weight 2
        if top_c >= min_assoc and top_c >= ratio * max_other:
            suggestions.append(
                {
                    "keyword": tok,
                    "suggested_labels": top_lab,
                    "suggested_weight": 2,
                    "freq_unclassified": freq,
                    **{f"count_{lab}": counts[lab] for lab in labels},
                }
            )
            continue

        # Two-label signal -> weight 1 in both labels
        if top_c >= min_assoc and top2_c >= min_assoc and top2_c >= ratio * max_rest:
            suggestions.append(
                {
                    "keyword": tok,
                    "suggested_labels": f"{top_lab} || {top2_lab}",
                    "suggested_weight": 1,
                    "freq_unclassified": freq,
                    **{f"count_{lab}": counts[lab] for lab in labels},
                }
            )

    sug_df = pd.DataFrame(suggestions)
    if not sug_df.empty:
        sug_df.sort_values(["suggested_weight", "freq_unclassified"], ascending=[False, False], inplace=True)
    return sug_df


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--episodes", default=str(DEFAULT_EPISODES_PATH), help="Path to All_episodes_with_messages.csv")
    ap.add_argument(
        "--dictionary",
        default=str(DEFAULT_DICT_XLSX if DEFAULT_DICT_XLSX.exists() else DEFAULT_DICT_CSV),
        help="Path to dictionary.xlsx or dictionary.csv",
    )
    ap.add_argument("--out_dir", default=str(DEFAULT_OUT_DIR), help="Output directory")
    ap.add_argument("--min_score", type=float, default=1.0, help="Minimum label score to classify")
    ap.add_argument("--multi_ratio", type=float, default=0.8, help="Keep additional labels if score >= top*multi_ratio")
    ap.add_argument("--top_n_candidates", type=int, default=200)
    ap.add_argument("--min_candidate_freq", type=int, default=5)
    args = ap.parse_args()

    episodes_path = Path(args.episodes)
    dict_path = Path(args.dictionary)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if not episodes_path.exists():
        raise FileNotFoundError(f"Episodes CSV not found: {episodes_path}")
    if not dict_path.exists():
        raise FileNotFoundError(f"Dictionary file not found: {dict_path}")

    df = pd.read_csv(episodes_path)
    dict_terms = load_dictionary_wide(dict_path)

    # Classify start/end
    start_labels, start_conf, start_kw = [], [], []
    end_labels, end_conf, end_kw = [], [], []

    for _, row in df.iterrows():
        stext = combine_fields(row, START_FIELDS)
        sc, sm = classify_text(stext, dict_terms)
        lab, conf, kw = assign_labels_multi(sc, sm, min_score=args.min_score, multi_ratio=args.multi_ratio)
        start_labels.append(lab)
        start_conf.append(conf)
        start_kw.append(kw)

        etext = combine_fields(row, END_FIELDS)
        ec, em = classify_text(etext, dict_terms)
        lab2, conf2, kw2 = assign_labels_multi(ec, em, min_score=args.min_score, multi_ratio=args.multi_ratio)
        end_labels.append(lab2)
        end_conf.append(conf2)
        end_kw.append(kw2)

    df["start_intent_labels"] = start_labels
    df["start_intent_confidence"] = start_conf
    df["start_intent_keywords"] = start_kw
    df["end_intent_labels"] = end_labels
    df["end_intent_confidence"] = end_conf
    df["end_intent_keywords"] = end_kw

    out_eps = out_dir / "episodes_with_intentions.csv"
    df.to_csv(out_eps, index=False, encoding="utf-8")

    # Candidate suggestions for expanding the dictionary
    start_sug = build_candidate_suggestions(
        df, START_FIELDS, "start_intent_labels", dict_terms,
        top_n=args.top_n_candidates, min_freq=args.min_candidate_freq
    )
    end_sug = build_candidate_suggestions(
        df, END_FIELDS, "end_intent_labels", dict_terms,
        top_n=args.top_n_candidates, min_freq=args.min_candidate_freq
    )

    (out_dir / "start_intent_candidate_keywords.csv").write_text(
        start_sug.to_csv(index=False, encoding="utf-8"), encoding="utf-8"
    )
    (out_dir / "end_intent_candidate_keywords.csv").write_text(
        end_sug.to_csv(index=False, encoding="utf-8"), encoding="utf-8"
    )

    # Console summary
    print("[ok] wrote:", out_eps)
    print("\nStart label distribution:")
    print(df["start_intent_labels"].value_counts(dropna=False).head(25))
    print("\nEnd label distribution:")
    print(df["end_intent_labels"].value_counts(dropna=False).head(25))
    print("\n[ok] wrote candidate keyword suggestions for dictionary expansion.")
    print("[info] out_dir:", out_dir)


if __name__ == "__main__":
    main()
